# 🎮 Ultimate Tic-Tac-Toe — IA Minimax + Alpha-Beta

**Projet Battle IA — Fondements de l'IA**  
Algorithme : **Minimax avec élagage Alpha-Beta** (cours CMO4)  
Format des coups : **colonne puis ligne** (1–9)

| Section | Contenu |
|---|---|
| 1 | 📦 Clonage du repo + imports |
| 2 | ⚙️ Configuration de l'IA |
| 3 | 🧪 Tests de validation |
| 4 | ⚔️ **Battle inter-Colab** (combat officiel) |
| 5 | 🤖 IA vs IA (calibrage) |
| 6 | 🧑 Humain vs IA |

> ⚠️ Exécutez **toujours** les sections 1 et 2 avant toute autre.

## 1. 📦 Clonage du repo et imports
Cette cellule clone le dépôt GitHub et importe **exactement** les mêmes modules que le projet local.
Le code n'est pas recopié : tout changement sur le repo est automatiquement pris en compte.

In [1]:
import subprocess, sys, os

# ── Clonage du repo ──────────────────────────────────────────────────
REPO_URL  = 'https://github.com/richardghostr/ultimate_tictactoe'
REPO_DIR  = '/content/ultimate_tictactoe'

if os.path.exists(REPO_DIR):
    print('Repo déjà cloné — mise à jour...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    print('Clonage du repo...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

# ── Ajout au path Python ─────────────────────────────────────────────
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print(f'Répertoire courant : {os.getcwd()}')

# ── Imports depuis le projet ─────────────────────────────────────────
from game.board   import Board, EMPTY, PLAYER_X, PLAYER_O, DRAW
from game.rules   import get_legal_moves, apply_move, check_global_win
from game.display import display_board, parse_human_move
from game.game    import Game
from ai.heuristic import evaluate, terminal_score, SCORE_GLOBAL_WIN
from ai.minimax   import minimax, get_best_move, get_best_move_timed
from ai.player    import AIPlayer, HumanPlayer

import time, math
print('\n✅ Tous les modules chargés depuis le repo.')
print(f'   Board         : {Board}')
print(f'   get_best_move : {get_best_move}')
print(f'   evaluate      : {evaluate}')

Repo déjà cloné — mise à jour...
Répertoire courant : c:\content\ultimate_tictactoe

✅ Tous les modules chargés depuis le repo.
   Board         : <class 'game.board.Board'>
   get_best_move : <function get_best_move at 0x0000018CB720F5E0>
   evaluate      : <function evaluate at 0x0000018CB720E6C0>


## 2. ⚙️ Configuration de l'IA
Modifiez **uniquement cette cellule** avant le combat.

In [2]:
# ════════════════════════════════════════════════════════════
#  PARAMÈTRES — adaptez avant chaque combat
# ════════════════════════════════════════════════════════════

NOM_IA     = 'MonIA'   # nom affiché pendant les combats

USE_TIMED  = True      # True  → approfondissement itératif (recommandé)
                       # False → profondeur fixe

TIME_LIMIT = 5.0       # secondes max par coup (recommandé : 2-6s, max 10s)
MAX_DEPTH  = 4         # profondeur max pour l'approfondissement itératif
DEPTH_FIXE = 4         # profondeur utilisée si USE_TIMED = False

# ── Fonction principale : appel à l'IA du projet ─────────────────────
def mon_ia_joue(board):
    """
    Retourne le meilleur coup pour l'état `board`.
    Utilise get_best_move_timed (ou get_best_move) du vrai projet.
    """
    if USE_TIMED:
        return get_best_move_timed(
            board,
            time_limit  = TIME_LIMIT,
            max_depth   = MAX_DEPTH,
            root_player = board.current_player
        )
    else:
        return get_best_move(
            board,
            depth       = DEPTH_FIXE,
            root_player = board.current_player
        )

print(f'✅ IA configurée : {NOM_IA}')
print(f'   Mode    : {"Timed (" + str(TIME_LIMIT) + "s, max_depth=" + str(MAX_DEPTH) + ")" if USE_TIMED else "Profondeur fixe = " + str(DEPTH_FIXE)}')
print(f'   Modules : get_best_move_timed={get_best_move_timed.__module__}')

✅ IA configurée : MonIA
   Mode    : Timed (5.0s, max_depth=4)
   Modules : get_best_move_timed=ai.minimax


## 3. 🧪 Tests de validation
Exécutez avant chaque combat pour s'assurer que tout fonctionne.

In [3]:
# ════════════════════════════════════════════════════════════
#  TESTS — vérifie l'intégrité du projet
# ════════════════════════════════════════════════════════════
import traceback
passed = failed = 0

def check(name, cond, msg=''):
    global passed, failed
    if cond:
        print(f'  ✅ {name}')
        passed += 1
    else:
        print(f'  ❌ {name} — {msg}')
        failed += 1

print('── Tests moteur de jeu ─────────────────────')
b = Board()
check('81 coups légaux au départ',      len(get_legal_moves(b)) == 81)

b2 = apply_move(b, (1,1,1,1))
check('apply_move immuable',            b.cells[1][1][1][1] == EMPTY)
check('Symbole correct après coup',     b2.cells[1][1][1][1] == PLAYER_X)
check('active_grid → (1,1)',            b2.active_grid == (1,1))
check('8 coups légaux après centre',    len(get_legal_moves(b2)) == 8)
check('Joueur suivant = O',             b2.current_player == PLAYER_O)

print('\n── Tests heuristique ───────────────────────')
check('Plateau vide → score = 0',       evaluate(b, PLAYER_X) == 0)
b3 = Board()
b3.cells[0][0][0][0] = b3.cells[0][0][1][0] = b3.cells[0][0][2][0] = PLAYER_X
from game.rules import _update_local_winner, _update_global_winner
_update_local_winner(b3, 0, 0)
check('Victoire locale X détectée',     b3.local_winner[0][0] == PLAYER_X)
check('Score > 0 après victoire locale',evaluate(b3, PLAYER_X) > 0)
b_win = Board(); b_win.global_winner = PLAYER_X
check('Score terminal victoire = WIN',  evaluate(b_win, PLAYER_X) == SCORE_GLOBAL_WIN)
check('Score terminal défaite = -WIN',  evaluate(b_win, PLAYER_O) == -SCORE_GLOBAL_WIN)

print('\n── Tests Minimax / IA ──────────────────────')
t0   = time.time()
move = mon_ia_joue(b)
dt   = time.time() - t0
check('IA retourne coup légal',         move in get_legal_moves(b))
check(f'Temps IA < TIME_LIMIT ({dt:.2f}s)', dt < TIME_LIMIT + 1.0)

print('\n── Test parse_human_move ───────────────────')
# Centre global = colonne 5, ligne 5 → (1,1,1,1)
b5 = Board()
m  = parse_human_move('5 5', b5)
check('parse_human_move("5 5") → (1,1,1,1)', m == (1,1,1,1))
m2 = parse_human_move('1 1', b5)
check('parse_human_move("1 1") → (0,0,0,0)', m2 == (0,0,0,0))
m3 = parse_human_move('9 9', b5)
check('parse_human_move("9 9") → (2,2,2,2)', m3 == (2,2,2,2))

print(f'\n{"═"*45}')
print(f'  {passed} tests réussis  /  {failed} échecs')
print('═'*45)
if failed == 0:
    print('  🚀 Prêt pour le combat !')
else:
    print('  ⚠️  Corrigez les erreurs avant de lancer un combat.')

── Tests moteur de jeu ─────────────────────
  ✅ 81 coups légaux au départ
  ✅ apply_move immuable
  ✅ Symbole correct après coup
  ✅ active_grid → (1,1)
  ✅ 8 coups légaux après centre
  ✅ Joueur suivant = O

── Tests heuristique ───────────────────────
  ✅ Plateau vide → score = 0
  ✅ Victoire locale X détectée
  ✅ Score > 0 après victoire locale
  ✅ Score terminal victoire = WIN
  ✅ Score terminal défaite = -WIN

── Tests Minimax / IA ──────────────────────
  ✅ IA retourne coup légal
  ✅ Temps IA < TIME_LIMIT (0.15s)

── Test parse_human_move ───────────────────
  ✅ parse_human_move("5 5") → (1,1,1,1)
  ✅ parse_human_move("1 1") → (0,0,0,0)
  ✅ parse_human_move("9 9") → (2,2,2,2)

═════════════════════════════════════════════
  16 tests réussis  /  0 échecs
═════════════════════════════════════════════
  🚀 Prêt pour le combat !


## 4. ⚔️ Battle inter-Colab — Combat officiel

**Principe** : un humain relaie les coups entre les deux Colabs.

### Étapes
1. Configurez `MON_ROLE` (X = vous commencez, O = l'adversaire commence)
2. Exécutez la cellule de config
3. Exécutez la cellule de combat → relancez pour chaque nouvelle partie
4. **Votre tour** → l'IA calcule, le coup s'affiche → vous le dictez
5. **Tour adverse** → saisissez le coup reçu (`colonne` puis `ligne`)

In [4]:
# ════════════════════════════════════════════════════════════
#  CONFIG COMBAT — modifiez avant chaque partie
# ════════════════════════════════════════════════════════════

MON_ROLE       = PLAYER_X    # PLAYER_X si vous commencez, PLAYER_O sinon
NOM_ADVERSAIRE = 'Adversaire'

print(f'⚙️  {NOM_IA} joue : {"X (commence)" if MON_ROLE==PLAYER_X else "O (second)"}')
print(f'   Adversaire : {NOM_ADVERSAIRE}')

⚙️  MonIA joue : X (commence)
   Adversaire : Adversaire


In [ ]:
# ════════════════════════════════════════════════════════════
#  COMBAT OFFICIEL — relancez cette cellule pour chaque partie
#  Version : instrumentée avec GameTimer pour afficher les chronos
# ════════════════════════════════════════════════════════════

board     = Board()
last_move = None
history   = []
timings   = []

# ── Instrumentation du timer ───────────────────────────────────
from game.clock import GameTimer
gt = GameTimer()
gt.start_total()
gt.start_for(board.current_player)

print('═' * 55)
print(f'  ⚔️  {NOM_IA} ({"X" if MON_ROLE==PLAYER_X else "O"}) vs {NOM_ADVERSAIRE}')
print('═' * 55)
display_board(board, timer=gt)

while not board.is_terminal():

    if board.current_player == MON_ROLE:
        # ── Mon tour : l'IA du projet calcule ────────────────────
        print(f'\n🤖 {NOM_IA} réfléchit...')
        t0   = time.time()
        move = mon_ia_joue(board)          # ← appel au vrai projet
        dt   = time.time() - t0
        timings.append(dt)

        gr, gc, lr, lc = move
        col_out = gc * 3 + lc + 1
        row_out = gr * 3 + lr + 1

        print()
        print('  ┌──────────────────────────────────────┐')
        print(f'  │  MON COUP → colonne {col_out}, ligne {row_out}           │')
        print(f'  │  Temps de réflexion : {dt:.2f}s             │')
        print('  └──────────────────────────────────────┘')
        print('  👉 Dictez ce coup à votre adversaire.')

    else:
        # ── Tour adverse : saisie manuelle du coup reçu ──────────
        ag = board.active_grid
        if ag:
            print(f'\n⌨️  Coup adversaire (zone : sous-grille ({ag[0]+1},{ag[1]+1}))')
        else:
            print(f'\n⌨️  Coup adversaire (zone libre)')

        move = None
        while move is None:
            raw  = input('    colonne (1-9) ligne (1-9)  ex: "5 3" : ').strip()
            move = parse_human_move(raw, board)
            if move is None:
                print('    ⚠️  Coup invalide (case prise ou hors zone) — réessayez.')

    # ── Arrêter le timer pour le joueur courant juste avant d'appliquer le coup ──
    gt.stop_for(board.current_player)

    board     = apply_move(board, move)
    last_move = move
    history.append(move)

    # ── Démarrer / basculer le timer pour le joueur suivant (si la partie continue) ──
    if not board.is_terminal():
        gt.switch_to(board.current_player)

    display_board(board, last_move=last_move, timer=gt)

# ── Résultat ─────────────────────────────────────────────────────────
gt.stop_all()
print('\n' + '═' * 55)
w = board.global_winner
if   w == MON_ROLE:  print(f'  🏆 VICTOIRE de {NOM_IA} !')
elif w == DRAW:       print('  🤝 MATCH NUL')
else:                 print(f'  💀 Défaite de {NOM_IA}.')
print(f'  Coups joués : {board.move_count}')
# ── Affichage des temps ──────────────────────────────────────────────
print(f'  Temps total : {gt.total_time():.2f}s | X : {gt.get_player_time(PLAYER_X):.2f}s | O : {gt.get_player_time(PLAYER_O):.2f}s')
if timings:
    print(f'  Temps IA  — moy : {sum(timings)/len(timings):.2f}s | max : {max(timings):.2f}s')
print('═' * 55)

═══════════════════════════════════════════════════════
  ⚔️  MonIA (X) vs Adversaire
═══════════════════════════════════════════════════════

   1  2  3     4  5  6     7  8  9 

1  . | . | .  ||  . | . | .  ||  . | . | . 
2  . | . | .  ||  . | . | .  ||  . | . | . 
3  . | . | .  ||  . | . | .  ||  . | . | . 
4  . | . | .  ||  . | . | .  ||  . | . | . 
5  . | . | .  ||  . | . | .  ||  . | . | . 
6  . | . | .  ||  . | . | .  ||  . | . | . 
7  . | . | .  ||  . | . | .  ||  . | . | . 
8  . | . | .  ||  . | . | .  ||  . | . | . 
9  . | . | .  ||  . | . | .  ||  . | . | . 

   1  2  3     4  5  6     7  8  9 
  Tour du joueur X | Zone : au choix | Coups : 0 | Temps total: 0.00s | X: 0.00s | O: 0.00s


🤖 MonIA réfléchit...

  ┌──────────────────────────────────────┐
  │  MON COUP → colonne 2, ligne 2           │
  │  Temps de réflexion : 0.01s             │
  └──────────────────────────────────────┘
  👉 Dictez ce coup à votre adversaire.

   1  2  3     4  5  6     7  8  9 

1  . | . | .  |

## 5. 🤖 IA vs IA — Calibrage
Utilisez cette section pour mesurer les performances et calibrer la profondeur/temps avant le combat.

In [ ]:
# ════════════════════════════════════════════════════════════
#  IA vs IA — utilise directement AIPlayer du projet
# ════════════════════════════════════════════════════════════

# Configuration des deux IA
IA1_TIME_LIMIT = TIME_LIMIT   # votre IA (approfondissement itératif)
IA1_MAX_DEPTH  = MAX_DEPTH
IA2_DEPTH      = 3            # IA adversaire simulée (profondeur fixe)
SHOW           = True         # False = beaucoup plus rapide

ia1 = AIPlayer(name=NOM_IA,      time_limit=IA1_TIME_LIMIT, max_depth=IA1_MAX_DEPTH)
ia2 = AIPlayer(name='IA-Adverse', depth=IA2_DEPTH)

scores = {ia1.name: 0, ia2.name: 0}
all_moves = []

for partie, (px, po) in enumerate([(ia1, ia2), (ia2, ia1)], 1):
    print(f'\n── Partie {partie} : {px.name} (X) vs {po.name} (O) ──')
    g = Game(px, po, show_board=SHOW)
    g.run()
    stats = g.get_stats()
    all_moves.append(stats['move_count'])
    w = stats['winner']
    sym = {PLAYER_X:'X', PLAYER_O:'O', DRAW:'Nul'}
    print(f'  Résultat : {sym[w]} en {stats["move_count"]} coups | '
          f'moy {stats["avg_time"]:.2f}s | max {stats["max_time"]:.2f}s')

# Score DVO
moy = sum(all_moves) / len(all_moves)
print('\n' + '═'*50)
print('  SCORE DVO (barème du projet)')
print('  Victoire rapide=4 | lente=3 | Nul rapide=1 | Défaite=0')

# Recalcul depuis les stats du Game
# (simplification : on relit les résultats des deux parties)
g1 = Game(ia1, ia2, show_board=False); r1 = g1.run(); s1 = g1.get_stats()
g2 = Game(ia2, ia1, show_board=False); r2 = g2.run(); s2 = g2.get_stats()

def pts(result, side, moves, moy):
    rapid = moves <= moy
    if result == side:   return 4 if rapid else 3
    elif result == DRAW: return 1 if rapid else 0
    else:                return 1 if rapid else 0

moy2 = (s1['move_count'] + s2['move_count']) / 2
sc1  = pts(r1, PLAYER_X, s1['move_count'], moy2) + pts(r2, PLAYER_O, s2['move_count'], moy2)
sc2  = pts(r1, PLAYER_O, s1['move_count'], moy2) + pts(r2, PLAYER_X, s2['move_count'], moy2)

print(f'\n  {ia1.name} : {sc1} pts  |  {ia2.name} : {sc2} pts')
print('═'*50)

## 6. 🧑 Humain vs IA — Entraînement
Jouez contre votre propre IA pour tester son niveau.

In [ ]:
# ════════════════════════════════════════════════════════════
#  HUMAIN vs IA — utilise Game + HumanPlayer du projet
# ════════════════════════════════════════════════════════════

HUMAIN_COMMENCE = True    # True = humain joue X (commence)

humain = HumanPlayer(name='Vous')
ia     = AIPlayer(name=NOM_IA, time_limit=TIME_LIMIT, max_depth=MAX_DEPTH)

if HUMAIN_COMMENCE:
    game = Game(humain, ia, show_board=True)
else:
    game = Game(ia, humain, show_board=True)

result = game.run()
stats  = game.get_stats()

w_sym = {PLAYER_X:'X', PLAYER_O:'O', DRAW:'Match nul'}
print(f'Résultat final : {w_sym[result]}')
print(f'Coups : {stats["move_count"]} | Temps moy IA : {stats["avg_time"]:.2f}s')